# Лабораторная работа № 3. Задание № 2.
> Григорьев Никита M3338, Паховский Иван M3339

## Условие варианта

Распределение Лапласса с неизвестным параметром сдвига $\mu$ и единичным масштабирующим параметром; $\mu$; $\mu = 2$;

## Выкладки

$X_1, \dots, X_n \sim \text{Laplace}(\mu, 1)$, плотность $f(x) = \frac{1}{2} e^{-|x - \mu|}$.

$\mathbb{E}X_i = \mathbb{Med}X_i = \mu$

Хотим придумать центральную статистику, чтобы с помощью уже известного нам выражения $\mathbb{P}(q_{\frac{\alpha}{2}} \le G(X, \theta) \le q_{1-\frac{\alpha}{2}}) = 1-\alpha$ построить доверительный интервал для $\mu$.

### Способ 1. Через выборочное среднее.
$$\sqrt{n}(\overline{X} - \mu) \overset d\longrightarrow \mathcal{N}(0,1) \quad \text{по ЦПТ.}$$

То есть $\sqrt{n}(\overline{X} - \mu)$ является центральной статистикой.

Тогда доверительный интервал выглядит следующим образом:

$$\mathbb{P}(-\frac{1}{\sqrt{n}}q_{1-\frac{\alpha}{2}} + \overline{X} \le\mu\le -\frac{1}{\sqrt{n}}q_{\frac{\alpha}{2}} + \overline{X})$$

 ### Способ 2. Через выборочную медиану.

Воспользуемся **Теоремой об асимптотике средних членов вариационного ряда**:

$$\sqrt{n}\frac{X_{(\lceil np\rceil)}-q_p}{\sqrt{p(1-p)}}f(q_q) d\longrightarrow \mathcal{N}(0,1)$$

$$\sqrt{n}\frac{\hat{q_{\frac{1}{2}}}-\mu}{\sqrt{\frac{1}{2}(1-\frac{1}{2})}}f(\mu) d\longrightarrow \mathcal{N}(0,1)$$

В случае распределения Лапласа $\frac{f(\mu)}{\sqrt{\frac{1}{2}(1-\frac{1}{2})}} = 1$, тогда итоговый интервал похож на тот, который мы находили с помощью выборочного среднего:

$$\mathbb{P}(-\frac{1}{\sqrt{n}}q_{1-\frac{\alpha}{2}} + \hat{q_{\frac{1}{2}}} \le\mu\le -\frac{1}{\sqrt{n}}q_{\frac{\alpha}{2}} + \hat{q_{\frac{1}{2}}})$$


In [ ]:
import numpy as np
import scipy.stats

MU = 2
SCALE = 1

ALPHA = 0.1

SAMPLE_SIZES = [10, 100, 1000, 10000, 100000, 500000]
NUM_SAPLES = 5000

In [ ]:
def calc_confidence_interval_mean(X, alpha):
    return (- scipy.stats.norm.ppf(1 - alpha / 2) / np.sqrt(len(X)) + np.mean(X),
            - scipy.stats.norm.ppf(alpha / 2) / np.sqrt(len(X)) + np.mean(X))


def calc_confidence_interval_median(X, alpha):
    return (- scipy.stats.norm.ppf(1 - alpha / 2) / np.sqrt(len(X)) + np.median(X),
            - scipy.stats.norm.ppf(alpha / 2) / np.sqrt(len(X)) + np.median(X))

In [ ]:
def run(num_samples, one_sample_size, alpha):
    global MU, SCALE

    intervals_mean = []
    good_intervals_count_mean = 0
    bad = 0

    intervals_median = []
    good_intervals_count_median = 0

    for _ in range(num_samples):
        X = np.random.laplace(MU, SCALE, one_sample_size)

        mu_l_mean, mu_r_mean = calc_confidence_interval_mean(X, alpha)
        mu_l_median, mu_r_median = calc_confidence_interval_median(X, alpha)

        if MU >= mu_l_mean and MU <= mu_r_mean:
            good_intervals_count_mean += 1
        elif one_sample_size == 100000:
            print(f'({mu_l_mean}, {mu_r_mean})')

        if MU >= mu_l_median and MU <= mu_r_median:
            good_intervals_count_median += 1

        intervals_mean.append((mu_l_mean, mu_r_mean))
        intervals_median.append((mu_l_median, mu_r_median))

    return (intervals_mean, good_intervals_count_mean, intervals_median, good_intervals_count_median)

In [ ]:
for i in SAMPLE_SIZES:
    intervals_mean, good_intervals_mean, intervals_median, good_intervals_median = run(NUM_SAPLES, i, ALPHA)

    print(f'N = {i}')
    print(f'Good intervals for mean estimate : {good_intervals_mean}')
    print(f'Good intervals for mean estimate: {good_intervals_mean / NUM_SAPLES}')

    print(f'Good intervals for median estimate : {good_intervals_median}')
    print(f'Good intervals for median estimate: {good_intervals_median / NUM_SAPLES}')

    firsts_mean, seconds_mean = zip(*intervals_mean)
    result = (sum(firsts_mean) / len(intervals_mean), sum(seconds_mean) / len(intervals_mean))

    print(f'Mean interval_mean: ({result[0]}, {result[1]})')
    print(f'Mean interval_mean len: {result[1] - result[0]}')

    firsts_median, seconds_median = zip(*intervals_median)
    result = (sum(firsts_median) / len(intervals_median), sum(seconds_median) / len(intervals_median))

    print(f'Mean interval_median: ({result[0]}, {result[1]})')
    print(f'Mean interval_median len: {result[1] - result[0]}')

    print()